# Exercises 1 and 2: NumPy indexing/slicing and the Dataset class

In [1]:
import sys
from pathlib import Path

import numpy as np

# find the repository root (the folder that contains "datasets"), wherever the notebook is
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'datasets').is_dir())
for path in (ROOT / 'src', ROOT):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

DATA = ROOT / 'datasets'

from si.io.csv_file import read_csv
from si.data.dataset import Dataset

## 1.1 Load iris.csv
iris.csv has a header row and the class in the last column, so it is read with `read_csv(features=True, label=True)`.

In [2]:
iris = read_csv(str(DATA / 'iris' / 'iris.csv'), sep=',', features=True, label=True)
iris.shape(), iris.features, iris.label

((150, 4),
 ['sepal_length', 'sepal_width', 'petal_length', 'petal_width'],
 'class')

## 1.2 Penultimate independent variable
`X[:, -2]` takes every row (`:`) and the second-to-last column. Selecting one column with an integer removes that dimension, so the result is 1D.

In [3]:
penultimate = iris.X[:, -2]
print(iris.features[-2], penultimate.shape)

petal_length (150,)


## 1.3 Last 10 samples and the mean of each feature
`X[-10:]` takes the last 10 rows; `mean(axis=0)` collapses the rows and gives one mean per column (feature).

In [4]:
last_10 = iris.X[-10:]
print(last_10.shape)
dict(zip(iris.features, last_10.mean(axis=0).round(3)))

(10, 4)


{'sepal_length': np.float64(6.45),
 'sepal_width': np.float64(3.03),
 'petal_length': np.float64(5.33),
 'petal_width': np.float64(2.17)}

## 1.4 Samples with all features <= 6
`X <= 6` is a boolean matrix; `.all(axis=1)` keeps a row only if **every** column is True. The resulting 1D mask is used for boolean indexing.

In [5]:
mask = (iris.X <= 6).all(axis=1)
print('samples:', mask.sum())
iris.X[mask].shape

samples: 89


(89, 4)

## 1.5 Samples with a class different from 'Iris-setosa'

In [6]:
not_setosa = iris.y != 'Iris-setosa'
print('samples:', not_setosa.sum())
np.unique(iris.y[not_setosa])

samples: 100


array(['Iris-versicolor', 'Iris-virginica'], dtype=object)

## Exercise 2: dropna, fillna, remove_by_index
`iris_missing_data.csv` has NaN values. Each method changes the Dataset in place and returns `self`, so a fresh copy is loaded before each example.

In [7]:
path = str(DATA / 'iris' / 'iris_missing_data.csv')
ds = read_csv(path, features=True, label=True)
print('NaN in X:', np.isnan(ds.X).sum(), '| rows with NaN:', np.isnan(ds.X).any(axis=1).sum())

NaN in X: 17 | rows with NaN: 16


### 2.1 dropna
Removes every row with at least one NaN, and the matching entries of y.

In [8]:
ds = read_csv(path, features=True, label=True).dropna()
print(ds.shape(), len(ds.y), 'NaN left:', np.isnan(ds.X).sum())

(134, 4) 134 NaN left: 0


### 2.2 fillna
Replaces NaN by a number, or by the mean/median of the column (computed ignoring NaN with `np.nanmean`/`np.nanmedian`). No rows are removed.

In [9]:
for value in [0.0, 'mean', 'median']:
    ds = read_csv(path, features=True, label=True).fillna(value)
    print(f'{value!s:>6}: shape {ds.shape()}, NaN left: {np.isnan(ds.X).sum()}')

   0.0: shape (150, 4), NaN left: 0
  mean: shape (150, 4), NaN left: 0
median: shape (150, 4), NaN left: 0


### 2.3 remove_by_index
Removes one sample (row of X and entry of y) with `np.delete`.

In [10]:
ds = read_csv(path, features=True, label=True)
print('before:', ds.shape(), ds.X[0], ds.y[0])
ds.remove_by_index(0)
print('after: ', ds.shape(), ds.X[0], ds.y[0])

before: (150, 4) [5.1 3.5 1.4 0.2] Iris-setosa
after:  (149, 4) [4.9 3.  1.4 0.2] Iris-setosa
